# arXiv AI 논문 샘플 수집

arXiv API에서 인공지능 관련 논문을 약 150개 받아 JSONL로 저장합니다.
저장 파일은 프로젝트 루트 기준 `data/arxiv_ai_papers.jsonl`이며, 각 줄은 논문 1건의 JSON 객체입니다.

> arXiv API 이용 시 요청 사이에 3초 이상 간격을 둡니다. 이미 파일이 있으면 덮어씁니다.

In [1]:
from pathlib import Path
import json
import time
import urllib.parse
import urllib.request
import xml.etree.ElementTree as ET
import certifi
import ssl

TARGET_COUNT = 150
PAGE_SIZE = 50
REQUEST_INTERVAL_SECONDS = 3.0

# 노트북 위치와 무관하게 프로젝트 루트의 data 폴더에 저장
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'day32_지식그래프_구축_적재':
    PROJECT_ROOT = PROJECT_ROOT.parent
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'arxiv_ai_papers.jsonl'
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# AI 핵심 분야 카테고리와 검색어를 함께 사용
SEARCH_QUERY = '(cat:cs.AI OR cat:cs.LG OR cat:cs.CL OR cat:cs.CV OR cat:cs.RO)'
SEARCH_QUERY += ' AND (all:artificial intelligence OR all:machine learning OR all:deep learning)'
API_URL = 'https://export.arxiv.org/api/query'
print(f'저장 위치: {OUTPUT_PATH.resolve()}')

저장 위치: C:\Users\Playdata\Desktop\enkoa-practice-knowledge-graph\data\arxiv_ai_papers.jsonl


In [2]:
ATOM_NS = {'atom': 'http://www.w3.org/2005/Atom'}
ARXIV_NS = {'arxiv': 'http://arxiv.org/schemas/atom'}

def text_or_none(parent, path, namespaces=ATOM_NS):
    node = parent.find(path, namespaces)
    return node.text.strip() if node is not None and node.text else None

def parse_entry(entry):
    authors = []
    for author in entry.findall('atom:author', ATOM_NS):
        name = text_or_none(author, 'atom:name')
        if name:
            authors.append(name)
    categories = [node.attrib['term'] for node in entry.findall('atom:category', ATOM_NS) if 'term' in node.attrib]
    links = {link.attrib.get('rel', 'alternate'): link.attrib.get('href') for link in entry.findall('atom:link', ATOM_NS)}
    return {
        'id': text_or_none(entry, 'atom:id'),
        'title': ' '.join((text_or_none(entry, 'atom:title') or '').split()),
        'abstract': ' '.join((text_or_none(entry, 'atom:summary') or '').split()),
        'authors': authors,
        'categories': categories,
        'published': text_or_none(entry, 'atom:published'),
        'updated': text_or_none(entry, 'atom:updated'),
        'doi': text_or_none(entry, 'arxiv:doi', ARXIV_NS),
        'pdf_url': links.get('related') or next((v for k, v in links.items() if k == 'alternate'), None),
        'source': 'arxiv'
    }

def fetch_page(start=0, max_results=PAGE_SIZE):
    params = {
        "search_query": SEARCH_QUERY,
        "start": start,
        "max_results": max_results,
        "sortBy": "submittedDate",
        "sortOrder": "descending",
    }

    url = f"{API_URL}?{urllib.parse.urlencode(params)}"
    request = urllib.request.Request(
        url,
        headers={"User-Agent": "ai-paper-sample/1.0"}
    )

    ssl_context = ssl.create_default_context(
        cafile=certifi.where()
    )

    with urllib.request.urlopen(
        request,
        timeout=60,
        context=ssl_context
    ) as response:
        root = ET.fromstring(response.read())

    return [
        parse_entry(entry)
        for entry in root.findall("atom:entry", ATOM_NS)
    ]

In [3]:
papers = []
seen_ids = set()
for start in range(0, TARGET_COUNT, PAGE_SIZE):
    page = fetch_page(start=start, max_results=min(PAGE_SIZE, TARGET_COUNT - start))
    for paper in page:
        if paper['id'] and paper['id'] not in seen_ids:
            seen_ids.add(paper['id'])
            papers.append(paper)
    print(f'{len(papers)}개 수집 완료')
    if len(papers) >= TARGET_COUNT or len(page) < PAGE_SIZE:
        break
    time.sleep(REQUEST_INTERVAL_SECONDS)

papers = papers[:TARGET_COUNT]
with OUTPUT_PATH.open('w', encoding='utf-8') as file:
    for paper in papers:
        file.write(json.dumps(paper, ensure_ascii=False) + '\n')

print(f'완료: {len(papers)}개 논문을 {OUTPUT_PATH}에 저장했습니다.')

50개 수집 완료
100개 수집 완료
150개 수집 완료
완료: 150개 논문을 c:\Users\Playdata\Desktop\enkoa-practice-knowledge-graph\data\arxiv_ai_papers.jsonl에 저장했습니다.


In [4]:
# JSONL 저장 결과 확인
with OUTPUT_PATH.open(encoding='utf-8') as file:
    saved_papers = [json.loads(line) for line in file if line.strip()]

assert len(saved_papers) == len(papers)
assert all(p['id'] and p['title'] and p['abstract'] for p in saved_papers)
print(f'검증 완료: {len(saved_papers)}개 레코드')
display(saved_papers[0])

검증 완료: 150개 레코드


{'id': 'http://arxiv.org/abs/2609.09158v1',
 'title': 'TANGO: Humanoid Navigation in Cluttered Environments with a Whole-Body Vision-Language-Action Model',
 'abstract': 'We study the problem of navigating cluttered indoor environments with a humanoid robot. Unlike conventional methods that model navigation as a 2D path planning problem, humanoid traversal in cluttered environments requires continuous geometry-aware whole-body adaptation, including coordinated arm placement, torso adjustment, and gait modulation for collision-free movement through complex 3D spaces. We introduce TANGO, the first whole-body vision-language navigation framework for language-conditioned humanoid traversal in cluttered environments. Given a natural-language instruction and egocentric RGB observations, TANGO directly predicts 29-DoF joint-space actions for downstream whole-body control. We train TANGO entirely in simulation by synthesizing diverse collision-free traversal behaviors via global path planning,